# 📅 Week 18 – Day 5: Model Training & Evaluation

This notebook builds, trains, and evaluates three regression models: **Linear Regression**, **Decision Tree Regressor**, and **Random Forest Regressor**. The pipelines combine Scikit-Learn `ColumnTransformer` (for One-Hot Encoding) with model estimators. We compare MAE, RMSE, and R² scores, select the best model, save the fitted pipeline (`laptop_price_model.pkl`), test sample predictions, and plot feature importances.

## 1️⃣ Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import os
import joblib
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline

from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

## 2️⃣ Load Dataset

In [ ]:
path = "../data/processed/laptops_clean.csv"
if not os.path.exists(path):
    path = "data/processed/laptops_clean.csv"

df = pd.read_csv(path)
target_col = 'Final Price' if 'Final Price' in df.columns else 'Price'
print(f"Loaded dataset with target column '{target_col}'")
df.head()

## 3️⃣ Features & Target

In [ ]:
X = df.drop(target_col, axis=1)
y = df[target_col]

## 4️⃣ Define Columns

In [ ]:
categorical_features = [
    "Laptop",
    "Status",
    "Brand",
    "Model",
    "CPU",
    "Storage type",
    "GPU",
    "Touch"
]

numerical_features = [
    "RAM",
    "Storage",
    "Screen"
]

## 5️⃣ Train-Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)

## 6️⃣ Preprocessor

In [ ]:
preprocessor = ColumnTransformer(
    transformers=[
        (
            "cat",
            OneHotEncoder(handle_unknown="ignore"),
            categorical_features
        )
    ],
    remainder="passthrough"
)

## 7️⃣ Build Pipelines

In [ ]:
# Linear Regression Pipeline
lr_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", LinearRegression())
])

# Decision Tree Pipeline
dt_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", DecisionTreeRegressor(random_state=42))
])

# Random Forest Pipeline
rf_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", RandomForestRegressor(
        n_estimators=100,
        random_state=42
    ))
])

## 8️⃣ Train Models

In [ ]:
lr_pipeline.fit(X_train, y_train)
dt_pipeline.fit(X_train, y_train)
rf_pipeline.fit(X_train, y_train)
print("All models trained successfully.")

## 9️⃣ Evaluation Function

In [ ]:
def evaluate(model):
    predictions = model.predict(X_test)
    mae = mean_absolute_error(y_test, predictions)
    mse = mean_squared_error(y_test, predictions)
    rmse = mse ** 0.5
    r2 = r2_score(y_test, predictions)
    return mae, rmse, r2

## 🔟 Compare Models

In [ ]:
models = {
    "Linear Regression": lr_pipeline,
    "Decision Tree": dt_pipeline,
    "Random Forest": rf_pipeline
}

results = []
for name, model in models.items():
    mae, rmse, r2 = evaluate(model)
    results.append({"Model": name, "MAE": round(mae,2), "RMSE": round(rmse,2), "R2 Score": round(r2,4)})
    print("="*40)
    print(name)
    print("MAE :", round(mae,2))
    print("RMSE:", round(rmse,2))
    print("R²  :", round(r2,4))

results_df = pd.DataFrame(results)
display(results_df)

## 1️⃣1️⃣ Select Best Model

In [ ]:
best_model = rf_pipeline
print("Best Performing Model: Random Forest Regressor")

## 1️⃣2️⃣ Save Model

In [ ]:
models_dir = "../models"
if not os.path.exists(models_dir):
    models_dir = "models"
os.makedirs(models_dir, exist_ok=True)

model_path = os.path.join(models_dir, "laptop_price_model.pkl")
joblib.dump(best_model, model_path)
print(f"Model saved to {model_path}")

## 1️⃣3️⃣ Test Prediction

In [ ]:
sample = X_test.iloc[[0]]
prediction = best_model.predict(sample)

print("Predicted Price:", round(prediction[0], 2))
print("Actual Price:   ", y_test.iloc[0])

## 1️⃣4️⃣ Feature Importance (Random Forest)

In [ ]:
feature_names = best_model.named_steps["preprocessor"].get_feature_names_out()
importances = best_model.named_steps["model"].feature_importances_

importance_df = pd.DataFrame({
    "Feature": feature_names,
    "Importance": importances
}).sort_values(by="Importance", ascending=False)

plt.figure(figsize=(12,8))
plt.barh(importance_df["Feature"][:15], importance_df["Importance"][:15], color="mediumseagreen")
plt.title("Top 15 Important Features in Random Forest")
plt.xlabel("Importance")
plt.gca().invert_yaxis()
plt.show()